# Capstone — Prioritizing Pages for Editorial Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** We are building this capstone step by step. The completed steps prepare five features and a separate historical outcome target, split clients, fit logistic regression, and compare it with the frozen rule. The model has not beaten the frozen rule on the primary ranking metric.

Run the code cells in order. Python 3, pandas, NumPy, and DuckDB are needed. The March warehouse file is reused locally; the setup also supports a gated download with a securely configured Hugging Face token, as in Week 3. Generated data caches stay out of Git. The notebook and aggregate JSON receipt are safe to commit.

## 1. Question

**Can past search-performance signals rank pages by later impression-decline risk better than a transparent rule, helping an editor choose which pages to review first?**

An editor has limited review capacity. The output will be a ranked page list with measured reasons, supporting investigation and a possible refresh. A wrong recommendation can waste review time or encourage unnecessary changes; a missed candidate may leave a problem unattended. The outcome measures decline, not whether editing would help. Observed associations and decision-support recommendations are the intended claims.

## 2. Data — build the five-feature table

We use `fact_content_daily_performance`, March 2026, from the FlyRank warehouse release `v20260703`, pinned repository revision `50cbf7c3909d07be4d1b5906b4d09e882e5acbf2`.

**Raw row:** one report date × client × content item. **Prepared row:** one client/content item at the March 18 decision point.

| Dates | Role |
|---|---|
| March 1–7 | First week of past measurements |
| March 8–14 | Second week of past measurements |
| March 15–17 | Reporting buffer; excluded |
| March 18–31 | Later outcome evidence; target only |

Keep a past day only when `gsc_data_available IS TRUE` and impressions/clicks are observed and non-negative. Require all 14 past days and at least 100 past impressions. That defines the past-only scoring population. A page needs all 14 usable later days to enter the labeled training/evaluation population; future coverage is not a scoring input.

Only positive-impression days with a positive, observed position contribute to average position. Unavailable position remains missing; any imputation will be fitted on training clients in a later step. No GA4, AI-referral, product-score, text, or incompatible query-table fields enter these five features. June remains untouched. The three-day buffer is an explicit assumption because historical ingestion timestamps are unavailable.

In [1]:
# Install only missing libraries; no credentials appear in this command.
import importlib.util
import subprocess
import sys

packages = {"duckdb": "duckdb", "pandas": "pandas"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

MONTH = "2026-03"
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
FILENAME = f"fact_content_daily_performance/month={MONTH}/data_0.parquet"

# Find a repo root if present; otherwise use the current Colab folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/ml-intern-dataset-and-lane-guide.md").is_file()), Path.cwd())
local_candidates = [
    root / "data/warehouse" / f"month={MONTH}" / "data_0.parquet",
    root / "data/warehouse" / FILENAME,
]
month_file = next((p for p in local_candidates if p.is_file()), None)

if month_file is None:
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import get_token, hf_hub_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    token = token or get_token()
    if not token:
        raise RuntimeError("Warehouse access is required: set HF_TOKEN securely, "
                           "use a local Hugging Face login, or supply the March parquet file. "
                           "Do not paste a token into this notebook.")
    month_file = Path(hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=FILENAME,
        revision=REVISION, token=token, cache_dir=str(root / "data/warehouse_cache"),
    ))
    del token

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '1GB'")
temp_folder = root / "data/warehouse_cache/duckdb_tmp"
temp_folder.mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = ?", [str(temp_folder)])
# This is a local cached file, so repeat verification does not rescan Hugging Face.
con.read_parquet(str(month_file)).create_view("march_raw")
required_columns = {
    "report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
    "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position",
}
observed_columns = set(con.table("march_raw").columns)
assert required_columns <= observed_columns, "Warehouse schema differs; inspect before proceeding."
print("Loaded the cached March partition; required field names verified.")

Loaded the cached March partition; required field names verified.


### A. Combine daily records into one past-window row per page

The query below looks only at March 1–14. It sums impressions and clicks, calculates the two weekly impression totals, and collects valid position measurements. It never reads the later outcome window.

In [2]:
past_query = """
SELECT client_hash_id, content_hash_id, COUNT(*) AS feature_days,
       MIN(report_date) AS feature_start, MAX(report_date) AS feature_end,
       SUM(gsc_impressions) AS feature_impressions,
       SUM(gsc_clicks) AS feature_clicks,
       SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07') AS first_week_impressions,
       SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-08') AS second_week_impressions,
       SUM(gsc_avg_position * gsc_impressions) FILTER (
           WHERE gsc_impressions > 0 AND gsc_avg_position > 0) AS position_numerator,
       SUM(gsc_impressions) FILTER (
           WHERE gsc_impressions > 0 AND gsc_avg_position > 0) AS position_denominator
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING feature_days = 14 AND COUNT(DISTINCT report_date) = 14 AND feature_impressions >= 100
ORDER BY client_hash_id, content_hash_id
"""
past_pages = con.execute(past_query).df()
past_pages["decision_date"] = pd.Timestamp("2026-03-18")
assert not past_pages.duplicated(["client_hash_id", "content_hash_id", "decision_date"]).any()
assert past_pages.feature_days.eq(14).all()
assert pd.to_datetime(past_pages.feature_start).eq(pd.Timestamp("2026-03-01")).all()
assert pd.to_datetime(past_pages.feature_end).eq(pd.Timestamp("2026-03-14")).all()
assert past_pages.feature_impressions.ge(100).all()
print(f"Prepared {len(past_pages):,} past-only page rows across "
      f"{past_pages.client_hash_id.nunique()} clients.")
past_pages[["feature_impressions", "feature_clicks",
            "first_week_impressions", "second_week_impressions"]].head(5)

Prepared 60,534 past-only page rows across 33 clients.


,feature_impressions,feature_clicks,first_week_impressions,second_week_impressions
0,114.0,1.0,52.0,62.0
1,272.0,0.0,138.0,134.0
2,399.0,2.0,214.0,185.0
3,197.0,0.0,114.0,83.0
4,169.0,0.0,82.0,87.0


### B. Calculate the five inputs

`X_past` is a dataframe: rows are pages, columns are the five numbers a model could receive. IDs and future outcomes stay outside it.

| Feature | Calculation | Meaning |
|---|---|---|
| `log_impressions` | `log(1 + past impressions)` | Past search exposure, on a compressed numeric scale |
| `log_clicks` | `log(1 + past clicks)` | Click volume, on the same compressed scale |
| `ctr_pct` | `100 × past clicks / past impressions` | Percentage of exposure that became clicks |
| `weighted_position` | Impression-weighted average of valid past daily positions | Search-position context; lower is better |
| `prior_week_change_pct` | `100 × (week 2 − week 1) / week 1` | Whether visibility was already falling or rising |

Every calculation uses March 1–14 only. A zero first-week denominator becomes missing instead of an infinite percentage. Position is an aggregate approximation from daily averages; exact within-day weighting cannot be recovered. No imputation, scaling fit, or model fitting happens in this step.

In [3]:
FEATURE_COLUMNS = [
    "log_impressions", "log_clicks", "ctr_pct",
    "weighted_position", "prior_week_change_pct",
]
X_past = pd.DataFrame({
    "log_impressions": np.log1p(past_pages.feature_impressions),
    "log_clicks": np.log1p(past_pages.feature_clicks),
    "ctr_pct": 100 * past_pages.feature_clicks / past_pages.feature_impressions,
    "weighted_position": past_pages.position_numerator / past_pages.position_denominator,
    "prior_week_change_pct": 100 * (
        past_pages.second_week_impressions - past_pages.first_week_impressions
    ) / past_pages.first_week_impressions.replace(0, np.nan),
}).replace([np.inf, -np.inf], np.nan)
assert list(X_past.columns) == FEATURE_COLUMNS and X_past.shape[1] == 5
print(f"Past-only feature table: {X_past.shape[0]:,} rows × {X_past.shape[1]} features.")
X_past.head(5).round(4)

Past-only feature table: 60,534 rows × 5 features.


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct
0,4.7449,0.6931,0.8772,12.4211,19.2308
1,5.6095,0.0000,0.0000,12.4669,-2.8986
2,5.9915,1.0986,0.5013,11.7068,-13.5514
3,5.2883,0.0000,0.0000,8.9188,-27.1930
4,5.1358,0.0000,0.0000,18.5562,6.0976


### C. Build the answer column separately

`y` is the answer the model will learn from: **1** if impressions in March 18–31 fell by more than 20% relative to March 1–14; **0** otherwise. These are equal-length 14-day windows. The later counts never become input features.

Not all pages have a complete later window. We retain all eligible past pages in `X_past` for scoring, while `X` and `y` contain only pages with a measurable complete outcome. This is a retrospective complete-case subset and can bias evaluation toward better-tracked pages. `page_context` stores identifiers and the decision date separately for grouping and recommendations.

In [4]:
outcome_query = """
SELECT client_hash_id, content_hash_id, COUNT(*) AS outcome_days,
       MIN(report_date) AS outcome_start, MAX(report_date) AS outcome_end,
       SUM(gsc_impressions) AS outcome_impressions
FROM march_raw
WHERE report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
  AND gsc_impressions >= 0 AND gsc_clicks >= 0
GROUP BY client_hash_id, content_hash_id
HAVING outcome_days = 14 AND COUNT(DISTINCT report_date) = 14
"""
later_outcomes = con.execute(outcome_query).df()
page_data = past_pages.merge(
    later_outcomes, on=["client_hash_id", "content_hash_id"],
    how="left", validate="one_to_one", sort=False,
)
# Prove that merging target evidence did not reorder the past feature rows.
assert page_data.content_hash_id.tolist() == past_pages.content_hash_id.tolist()
assert page_data.client_hash_id.tolist() == past_pages.client_hash_id.tolist()
has_target = page_data.outcome_days.eq(14)
labeled_pages = page_data.loc[has_target].reset_index(drop=True)
X = X_past.loc[has_target].reset_index(drop=True).copy()
y = (
    labeled_pages.outcome_impressions.lt(0.8 * labeled_pages.feature_impressions)
    .astype("int64").rename("future_decline_proxy")
)
page_context = labeled_pages[
    ["client_hash_id", "content_hash_id", "decision_date"]
].copy()
groups = page_context.client_hash_id.copy()
assert len(X) == len(y) == len(page_context)
assert X.index.equals(y.index) and X.index.equals(page_context.index)
assert pd.to_datetime(labeled_pages.outcome_start).eq(pd.Timestamp("2026-03-18")).all()
assert pd.to_datetime(labeled_pages.outcome_end).eq(pd.Timestamp("2026-03-31")).all()
assert set(y.unique()) == {0, 1}
assert list(X.columns) == FEATURE_COLUMNS
assert not any(name in X.columns for name in ["client_hash_id", "content_hash_id",
               "future_decline_proxy", "outcome_impressions", "outcome_days"])
print(f"X: {len(X):,} rows × five features; y: {len(y):,} answers.")
print(f"Measured later decline: {int(y.sum()):,} pages ({y.mean():.2%}); "
      f"no such decline: {int(y.eq(0).sum()):,} pages.")
print(f"Clients: {groups.nunique()}; past-only pages without a complete target: "
      f"{int((~has_target).sum()):,}.")
X.head(5).round(4)

X: 57,459 rows × five features; y: 57,459 answers.
Measured later decline: 21,053 pages (36.64%); no such decline: 36,406 pages.
Clients: 32; past-only pages without a complete target: 3,075.


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct
0,4.7449,0.6931,0.8772,12.4211,19.2308
1,5.6095,0.0000,0.0000,12.4669,-2.8986
2,5.9915,1.0986,0.5013,11.7068,-13.5514
3,7.3645,2.7081,0.8872,7.3086,-63.3766
4,7.1678,0.0000,0.0000,56.0039,-15.6472


### D. Read one prepared row

The next small table shows original counts beside selected calculated values and the separate answer. It is a learning preview, **not** the model input table: the model receives only `X`'s five columns. The row number matches `X`, `y`, and `page_context`.

For example, CTR is a percentage: one click from 114 impressions is about 0.877%, not 87.7%. A positive past weekly change can still be followed by a later decline; a feature is a possible clue, not a guaranteed answer.

In [5]:
learning_preview = pd.DataFrame({
    "past_impressions": labeled_pages.feature_impressions,
    "past_clicks": labeled_pages.feature_clicks,
    "week1_impressions": labeled_pages.first_week_impressions,
    "week2_impressions": labeled_pages.second_week_impressions,
    "ctr_pct": X.ctr_pct,
    "prior_week_change_pct": X.prior_week_change_pct,
    "later_impressions_TARGET_ONLY": labeled_pages.outcome_impressions,
    "y_TARGET_ONLY": y,
})
learning_preview.head(5).round(4)

,past_impressions,past_clicks,week1_impressions,week2_impressions,ctr_pct,prior_week_change_pct,later_impressions_TARGET_ONLY,y_TARGET_ONLY
0,114.0,1.0,52.0,62.0,0.8772,19.2308,189.0,0
1,272.0,0.0,138.0,134.0,0.0000,-2.8986,158.0,1
2,399.0,2.0,214.0,185.0,0.5013,-13.5514,233.0,1
3,1578.0,14.0,1155.0,423.0,0.8872,-63.3766,1801.0,0
4,1296.0,0.0,703.0,593.0,0.0000,-15.6472,601.0,1


### E. Check and save this preparation step

We check missing values, variation, alignment, and consistency with the existing Week 4 population. The three CSV files are local caches in the repo's ignored work area; they must not be committed. The JSON receipt contains aggregate information, making it suitable to commit with the notebook.

The data step prepares inputs; splitting clients, fitting any preprocessing, training a model, and comparing it with the frozen rule are the next steps.

In [6]:
import json
import hashlib

quality = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "missing_rows": [int(X[c].isna().sum()) for c in FEATURE_COLUMNS],
    "distinct_values": [int(X[c].nunique()) for c in FEATURE_COLUMNS],
})
assert quality.distinct_values.gt(1).all(), "A feature is constant; investigate before training."
assert not np.isinf(X.to_numpy(dtype=float)).any()

output_dir = root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
X.to_csv(output_dir / "capstone_features.csv", index=False)
y.to_frame().to_csv(output_dir / "capstone_target.csv", index=False)
page_context.to_csv(output_dir / "capstone_page_context.csv", index=False)

baseline_path = output_dir / "w04_baseline_metrics.json"
if baseline_path.is_file():
    baseline_receipt = json.loads(baseline_path.read_text(encoding="utf-8"))
    assert baseline_receipt["labeled_pages"] == len(X)
    assert baseline_receipt["past_only_scoring_pages"] == len(X_past)

identity_rows = pd.concat([page_context, y], axis=1).to_csv(index=False)
feature_receipt = {
    "step": "five-feature table; model training pending",
    "warehouse_revision": REVISION, "month": MONTH,
    "decision_date": "2026-03-18",
    "feature_window": ["2026-03-01", "2026-03-14"],
    "outcome_window": ["2026-03-18", "2026-03-31"],
    "minimum_feature_impressions": 100, "required_days_per_window": 14,
    "feature_names": FEATURE_COLUMNS,
    "past_only_pages": len(X_past), "labeled_pages": len(X),
    "clients": int(groups.nunique()), "positive_targets": int(y.sum()),
    "negative_targets": int(y.eq(0).sum()),
    "pages_without_complete_target": int((~has_target).sum()),
    "quality": quality.to_dict(orient="records"),
    "identity_and_target_sha256": hashlib.sha256(identity_rows.encode("utf-8")).hexdigest(),
    "notes": "Targets and IDs are separate from X. Missing position stays missing until train-only preprocessing. March complete-case slice; historical ingestion times unknown.",
}
(output_dir / "capstone_feature_table.json").write_text(
    json.dumps(feature_receipt, indent=2, allow_nan=False) + chr(10), encoding="utf-8"
)
print("Saved three local CSV caches and the aggregate capstone_feature_table.json receipt.")
print("All five features have variation; target and context alignment verified.")
quality

Saved three local CSV caches and the aggregate capstone_feature_table.json receipt.
All five features have variation; target and context alignment verified.


,feature,missing_rows,distinct_values
0,log_impressions,0,8318
1,log_clicks,0,223
2,ctr_pct,0,16224
3,weighted_position,0,55457
4,prior_week_change_pct,0,39897


## 3. Methodology — separate learning from evaluation

The **training set** is the collection of pages the model can learn from. The **test set** is a separate collection for comparing the model with the frozen Week 4 rule. Split whole clients: pages from the same client share circumstances, so mixing their pages between learning and evaluation can make performance look better than it generalizes.

Use `GroupShuffleSplit`, client as the group, seed 42, and a 25% test-client fraction. This reproduces Week 4's split. The fraction refers to clients, not pages: inventories are unequal. IDs choose the split and later identify recommendations; they never become features. We explicitly check client overlap, row overlap, coverage, feature/target alignment, and the baseline's test-population fingerprint.

**Scope of this check:** features precede outcomes, but both groups use the same March decision date. This evaluates generalization to clients absent from model training, not generalization to a later calendar period. March test outcomes were already examined in earlier assignments, so this is a reused evaluation set, not a blind sealed test. June remains untouched. We will select model settings within training clients only; scaling, imputation, and model fitting occur only after this split in the cells below.

**Success metric, fixed before fitting:** mean client Precision@20: rank the pages for each test client, count how many of its first 20 later declined, then average across clients with at least 20 eligible pages. The model and frozen rule must use the same rows and tie policy (content ID for equal scores). A client with fewer than 20 eligible pages is excluded from this metric and counted explicitly. This measures useful prioritization of observed decline risk, not whether a refresh will help.


In [7]:
from sklearn.model_selection import GroupShuffleSplit

train_idx, test_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
).split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
context_train = page_context.iloc[train_idx].copy()
context_test = page_context.iloc[test_idx].copy()
train_clients = set(context_train.client_hash_id)
test_clients = set(context_test.client_hash_id)

assert train_clients.isdisjoint(test_clients), "A client leaked across the split."
assert set(train_idx).isdisjoint(set(test_idx)), "A page leaked across the split."
assert set(train_idx) | set(test_idx) == set(range(len(X)))
for inputs, targets, context in [(X_train, y_train, context_train),
                                 (X_test, y_test, context_test)]:
    assert inputs.index.equals(targets.index) and inputs.index.equals(context.index)
    assert list(inputs.columns) == FEATURE_COLUMNS and inputs.shape[1] == 5
    assert set(targets.unique()) == {0, 1}

# Match the previously recorded baseline without calculating new model performance.
test_identity = pd.concat([context_test[["client_hash_id", "content_hash_id"]], y_test], axis=1)
test_identity_csv = test_identity.sort_values(["client_hash_id", "content_hash_id"]).to_csv(index=False)
test_fingerprint = hashlib.sha256(test_identity_csv.encode("utf-8")).hexdigest()
if baseline_path.is_file():
    assert sorted(train_clients) == baseline_receipt["train_clients"]
    assert sorted(test_clients) == baseline_receipt["test_clients"]
    assert test_fingerprint == baseline_receipt["test_population_sha256"]

split_summary = pd.DataFrame([
    {"set": "training", "clients": len(train_clients), "pages": len(X_train),
     "declined_pages": int(y_train.sum()), "decline_rate_pct": 100 * y_train.mean()},
    {"set": "testing", "clients": len(test_clients), "pages": len(X_test),
     "declined_pages": int(y_test.sum()), "decline_rate_pct": 100 * y_test.mean()},
])
test_client_sizes = context_test.groupby("client_hash_id").size()
eligible_metric_clients = int(test_client_sizes.ge(20).sum())
small_test_clients = int(test_client_sizes.lt(20).sum())

split_receipt = {
    "step": "client split completed; model training pending",
    "group": "client_hash_id", "method": "GroupShuffleSplit",
    "seed": 42, "test_client_fraction": 0.25,
    "client_overlap": 0, "row_overlap": 0,
    "feature_names": FEATURE_COLUMNS,
    "summary": split_summary.to_dict(orient="records"),
    "train_clients": sorted(train_clients), "test_clients": sorted(test_clients),
    "test_population_sha256": test_fingerprint,
    "primary_metric": "mean client Precision@20",
    "metric_k": 20, "eligible_test_clients": eligible_metric_clients,
    "small_test_clients_excluded": small_test_clients,
    "limitations": "Same March cutoff; reused evaluation clients and outcomes, not a blind temporal holdout. June untouched. Settings must be chosen within training clients.",
}
split_receipt_path = output_dir / "capstone_split.json"
split_receipt_path.write_text(json.dumps(split_receipt, indent=2, allow_nan=False) + chr(10),
                              encoding="utf-8")
print("Client overlap: 0; page overlap: 0; all labeled pages assigned exactly once.")
print("Split and test population match the frozen Week 4 baseline.")
print(f"Precision@20 will use {eligible_metric_clients} test clients; "
      f"{small_test_clients} test client has fewer than 20 eligible pages.")
print("Prepared X_train, y_train, X_test, y_test. No model trained yet.")
split_summary.round(2)


Client overlap: 0; page overlap: 0; all labeled pages assigned exactly once.
Split and test population match the frozen Week 4 baseline.
Precision@20 will use 7 test clients; 1 test client has fewer than 20 eligible pages.
Prepared X_train, y_train, X_test, y_test. No model trained yet.


,set,clients,pages,declined_pages,decline_rate_pct
0,training,24,14070,4457,31.68
1,testing,8,43389,16596,38.25


### A. Fit the first model

The student wrote and ran `scripts/train_capstone_model.py` in small steps. The following cells carry the same training, validation, comparison, and error-review logic into the capstone notebook, so running this notebook from the warehouse reproduces the analysis.

Use logistic regression, `C=1`, `lbfgs`, at most 1,000 iterations, and seed 42. `SimpleImputer(strategy="median")` and `StandardScaler` form a pipeline with the classifier. They fit only on the training subset, and on each fold's training clients during cross-validation. There is currently no feature missingness, so the imputer changes no observed values. We use the default L2 regularization and do not tune settings against test scores. Class-1 scores rank review candidates; they are not calibrated probability guarantees.

In [8]:
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Reuse the page context and split built above, keeping IDs outside X.
context = page_context.copy()
OUTPUTS = output_dir
train_mask = context.client_hash_id.isin(train_clients)
test_mask = context.client_hash_id.isin(test_clients)
assert X_train.index.tolist() == X.index[train_mask].tolist()
assert X_test.index.tolist() == X.index[test_mask].tolist()

# Connect input preparation and the model.
model = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
    )),
])

# Learn using training pages and their answers only.
model.fit(X_train, y_train)

print("\nModel training complete.")
print("Model:", type(model.named_steps["classifier"]).__name__)
print("Number of input features:", model.n_features_in_)


Model training complete.
Model: LogisticRegression
Number of input features: 5


### B. Validate inside training clients

Four-fold `GroupKFold` leaves whole clients out. `cross_val_predict` clones the full pipeline for every fold; each training page gets one prediction from a model that did not train on its client. The original all-training-client model remains fitted for later evaluation. Mean client Precision@20 uses the 16 training clients with at least 20 eligible pages. The random reference is the equal-client mean decline rate over the same eligible clients. Pooled out-of-fold ROC-AUC is secondary; it pools predictions from different fitted folds and should not be compared directly with test AUC as if they were the same population.

In [9]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

# Keep client IDs outside the model inputs.
training_context = context.loc[
    train_mask, ["client_hash_id", "content_hash_id"]
].copy()

# Each prediction comes from a model that did not train on that client.
# cross_val_predict creates fresh copies of our entire pipeline.
cv_scores = cross_val_predict(
    model,
    X_train,
    y_train,
    groups=training_context["client_hash_id"],
    cv=GroupKFold(n_splits=4),
    method="predict_proba",
    n_jobs=1,
)[:, 1]

# Combine validation scores with identifiers and observed answers.
validation = training_context.copy()
validation["risk_score"] = cv_scores
validation["actual_decline"] = y_train

# Precision@20 requires at least 20 pages per client.
client_sizes = validation.groupby("client_hash_id")[
    "content_hash_id"
].transform("size")

eligible = validation.loc[client_sizes >= 20].copy()

# Rank within each client; resolve equal scores using content ID.
ranked = eligible.sort_values(
    ["client_hash_id", "risk_score", "content_hash_id"],
    ascending=[True, False, True],
)

top20 = ranked.groupby("client_hash_id").head(20)

# Give each eligible client equal weight.
precision20 = top20.groupby("client_hash_id")[
    "actual_decline"
].mean().mean()

random_precision20 = eligible.groupby("client_hash_id")[
    "actual_decline"
].mean().mean()

print("\nTRAINING-CLIENT CROSS-VALIDATION")
print("Eligible clients:", eligible["client_hash_id"].nunique())
print(f"Model mean Precision@20: {precision20:.1%}")
print(f"Expected random Precision@20: {random_precision20:.1%}")
print(f"ROC-AUC: {roc_auc_score(y_train, cv_scores):.3f}")

cv_summary = pd.DataFrame([{
    "method": "4-fold client-grouped validation",
    "eligible_clients": int(eligible.client_hash_id.nunique()),
    "precision_at_20_pct": float(100 * precision20),
    "expected_random_precision_at_20_pct": float(100 * random_precision20),
    "roc_auc": float(roc_auc_score(y_train, cv_scores)),
}])
cv_summary.round(3)



TRAINING-CLIENT CROSS-VALIDATION
Eligible clients: 16
Model mean Precision@20: 63.1%
Expected random Precision@20: 40.4%
ROC-AUC: 0.573


,method,eligible_clients,precision_at_20_pct,expected_random_precision_at_20_pct,roc_auc
0,4-fold client-grouped validation,16,63.125,40.409,0.573


## 4. Results — model versus frozen baseline

Evaluate the already-fitted model and the frozen Week 4 rule on the same 43,389 labeled test pages from eight clients. Precision@20 includes seven clients with at least 20 pages (140 recommendation slots); the eighth client is excluded only from that metric. ROC-AUC includes all test pages. Equal-score ties use content ID. The rule scores every eligible labeled page, including zero-score ties for this matched comparison; its exported action queue contains positive scores only.

The baseline gates require at least 300 past impressions and a past weekly change of at most −20%; the score is `log(1 + impressions) × (−weekly_change / 100)` when both gates pass, otherwise zero. These inputs are entirely pre-decision. We verify the original client sets, exact test identities/targets fingerprint, and saved baseline Precision@20.

In [10]:
import numpy as np

baseline = json.loads(
    (OUTPUTS / "w04_baseline_metrics.json").read_text(encoding="utf-8")
)

test_context = context.loc[
    test_mask, ["client_hash_id", "content_hash_id"]
].copy()

# Confirm that we are using the baseline's test clients and page count.
assert sorted(test_context["client_hash_id"].unique()) == baseline["test_clients"]
assert len(X_test) == baseline["test_pages"]

# Verify the exact test page identities and targets, not just their count.
test_identity = pd.concat([test_context, y_test], axis=1)
test_csv = test_identity.sort_values(["client_hash_id", "content_hash_id"]).to_csv(index=False)
assert hashlib.sha256(test_csv.encode("utf-8")).hexdigest() == baseline["test_population_sha256"]

# Our original model remains fitted on all training pages.
# Cross-validation used separate copies of it.
model_scores = model.predict_proba(X_test)[:, 1]

# Recover original integer impression counts from the log feature.
past_impressions = np.rint(np.expm1(X_test["log_impressions"]))
past_change = X_test["prior_week_change_pct"]

# Reproduce the frozen Week 4 rule.
rule_scores = np.where(
    (past_impressions >= baseline["rule_min_impressions"])
    & (past_change <= baseline["rule_max_prior_change_pct"]),
    np.log1p(past_impressions) * (-past_change / 100),
    0.0,
)

test_results = test_context.copy()
test_results["actual_decline"] = y_test
test_results["model_score"] = model_scores
test_results["rule_score"] = rule_scores

sizes = test_results.groupby("client_hash_id")[
    "content_hash_id"
].transform("size")

eligible_test = test_results.loc[sizes >= 20].copy()


def evaluate_ranking(score_column):
    ranked = eligible_test.sort_values(
        ["client_hash_id", score_column, "content_hash_id"],
        ascending=[True, False, True],
    )
    top20 = ranked.groupby("client_hash_id").head(20)

    return top20.groupby("client_hash_id")[
        "actual_decline"
    ].mean().mean()


comparison = pd.DataFrame([
    {
        "method": "Logistic regression",
        "precision_at_20_pct": 100 * evaluate_ranking("model_score"),
        "roc_auc": roc_auc_score(y_test, model_scores),
    },
    {
        "method": "Week 4 rule",
        "precision_at_20_pct": 100 * evaluate_ranking("rule_score"),
        "roc_auc": roc_auc_score(y_test, rule_scores),
    },
])

# Verify that the reproduced baseline matches its saved result.
assert np.isclose(
    evaluate_ranking("rule_score"),
    baseline["mean_client_precision_at_20"],
)

random_test_precision = eligible_test.groupby("client_hash_id")[
    "actual_decline"
].mean().mean()

print("\nTEST COMPARISON")
print(comparison.round(3).to_string(index=False))
print(f"Expected random Precision@20: {random_test_precision:.1%}")
print("Eligible test clients:", eligible_test["client_hash_id"].nunique())

# A random-ranking reference belongs in the same comparison table.
display_comparison = pd.concat([comparison, pd.DataFrame([{
    "method": "Expected random ranking",
    "precision_at_20_pct": 100 * random_test_precision,
    "roc_auc": 0.5,
}])], ignore_index=True)
display_comparison.round(3)



TEST COMPARISON
             method  precision_at_20_pct  roc_auc
Logistic regression               60.714    0.642
        Week 4 rule               61.429    0.564
Expected random Precision@20: 39.8%
Eligible test clients: 7


,method,precision_at_20_pct,roc_auc
0,Logistic regression,60.714,0.642
1,Week 4 rule,61.429,0.564
2,Expected random ranking,39.810,0.500


**Interpretation.** Logistic regression reaches 60.714% mean client Precision@20, while the rule reaches 61.429%, against a 39.810% random-ranking expectation. This is 85 versus 86 later-declining pages in 140 top-20 slots. The model has higher pooled ROC-AUC (0.642 versus 0.564), but did not beat the baseline on the primary metric fixed before training. The model's training-client cross-validation Precision@20 is 63.125% versus a 40.409% random expectation; its pooled out-of-fold AUC is 0.573.

We retain the rule as the provisional action method. This is an observed one-page difference, not established statistical superiority. Precision counts observed later decline, not editorial benefit. No extra model was selected or adjusted after seeing these test results.

### C. Read weights and three errors

Weights refer to standardized inputs and associations conditional on the other inputs. The largest absolute weights are click volume (negative), exposure (positive), and prior weekly change (negative). High exposure with few clicks and an earlier falling trend are plausible review clues. Clicks, exposure, and CTR are related by construction, so coefficients can redistribute that shared information. CTR's positive conditional coefficient is not evidence that improving CTR causes decline.

The three highest-scoring false positives within the model's top-20 lists all have zero clicks. Case 1 has only a mild prior weekly fall (−8.85%); a high-exposure/no-click pattern did not imply later decline. Case 2 was previously growing (+32.61%); the score can over-prioritize this pattern despite positive momentum. Case 3 already fell −38.48%, but did not meet the later >20% decline target; an earlier fall need not continue. Measurement quality, demand changes, or natural variation could explain these errors, but this slice does not identify their causes. The model's 0.82–0.85 scores are uncalibrated.

These are numeric record reviews, not article inspections. The printed examples omit identifiers, domains, URLs, and text.

In [11]:
# Model weights apply to standardized features.
weights = pd.DataFrame({
    "feature": X.columns,
    "weight": model.named_steps["classifier"].coef_[0],
})
weights["absolute_weight"] = weights["weight"].abs()

print("\nMODEL WEIGHTS")
print(
    weights.sort_values("absolute_weight", ascending=False)
    .round(4)
    .to_string(index=False)
)

# Find false positives within the model's top-20 recommendations.
model_top20 = (
    eligible_test.sort_values(
        ["client_hash_id", "model_score", "content_hash_id"],
        ascending=[True, False, True],
    )
    .groupby("client_hash_id")
    .head(20)
)

wrong_picks = (
    model_top20.loc[model_top20["actual_decline"].eq(0)]
    .nlargest(3, "model_score")
)

error_examples = X_test.loc[wrong_picks.index].copy()
error_examples["risk_score"] = wrong_picks["model_score"]
error_examples["actual_decline"] = wrong_picks["actual_decline"]

print("\nTHREE HIGH-SCORING FALSE POSITIVES")
print(error_examples.round(4).to_string(index=False))

error_examples.round(4)


MODEL WEIGHTS
              feature  weight  absolute_weight
           log_clicks -0.7439           0.7439
      log_impressions  0.5597           0.5597
prior_week_change_pct -0.5508           0.5508
              ctr_pct  0.1806           0.1806
    weighted_position  0.0791           0.0791

THREE HIGH-SCORING FALSE POSITIVES
 log_impressions  log_clicks  ctr_pct  weighted_position  prior_week_change_pct  risk_score  actual_decline
          9.8048         0.0      0.0            44.4338                -8.8511      0.8516               0
         10.3417         0.0      0.0            32.4991                32.6129      0.8500               0
          9.5946         0.0      0.0             4.4739               -38.4776      0.8243               0


,log_impressions,log_clicks,ctr_pct,weighted_position,prior_week_change_pct,risk_score,actual_decline
10591,9.8048,0.0,0.0,44.4338,-8.8511,0.8516,0
13732,10.3417,0.0,0.0,32.4991,32.6129,0.8500,0
25225,9.5946,0.0,0.0,4.4739,-38.4776,0.8243,0


### D. Save reproducibility receipts

The student's original `capstone_model_metrics.json` records Python 3.14 and scikit-learn 1.9. This notebook records its own runtime in `capstone_notebook_metrics.json`, preserving the original receipt. It checks that Precision@20 agrees and ROC-AUC agrees at the reported three decimals. The model binary is saved locally under ignored `data/models/`; the notebook, source script, and aggregate JSON receipts can be committed. No raw exports enter Git.

In [12]:
import sys
import sklearn

# Save the trained pipeline locally.
# data/ is ignored by Git, so the model file stays out of the repo.
model_folder = root / "data" / "models"
model_folder.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "pipeline": model,
        "feature_names": list(X.columns),
        "split_seed": 42,
    },
    model_folder / "capstone_notebook_model.joblib",
)

# Save aggregate results that we can commit.
metrics = {
    "model": "LogisticRegression",
    "execution_source": "capstone.ipynb",
    "model_parameters": model.named_steps["classifier"].get_params(),
    "python_version": sys.version.split()[0],
    "sklearn_version": sklearn.__version__,
    "feature_names": list(X.columns),
    "test_population_sha256": test_fingerprint,
    "training_pages": len(X_train),
    "test_pages": len(X_test),
    "primary_metric": "mean client Precision@20",
    "training_cross_validation": {
        "method": "4-fold GroupKFold by client",
        "precision_at_20": float(precision20),
        "expected_random_precision_at_20": float(random_precision20),
        "roc_auc": float(roc_auc_score(y_train, cv_scores)),
    },
    "test_comparison": comparison.to_dict(orient="records"),
    "test_expected_random_precision_at_20": float(random_test_precision),
    "eligible_test_clients": int(
        eligible_test["client_hash_id"].nunique()
    ),
    "standardized_feature_weights": {
        feature: float(weight)
        for feature, weight in zip(
            X.columns,
            model.named_steps["classifier"].coef_[0],
        )
    },
    "limitations": [
        "One March decision window, not a future-month evaluation.",
        "Test outcomes were previously examined in weekly assignments.",
        "Complete later coverage is required for evaluation.",
        "Decline prediction does not establish refresh benefit.",
        "Probability calibration has not been evaluated.",
    ],
}

metrics_path = OUTPUTS / "capstone_notebook_metrics.json"
metrics_path.write_text(
    json.dumps(metrics, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("\nSaved the trained model locally.")
print("Saved aggregate results:", metrics_path.name)

# Preserve the student's original receipt; compare headline values explicitly.
student_receipt_path = OUTPUTS / "capstone_model_metrics.json"
if student_receipt_path.is_file():
    student_receipt = json.loads(student_receipt_path.read_text(encoding="utf-8"))
    for current, original in zip(metrics["test_comparison"], student_receipt["test_comparison"]):
        assert current["method"] == original["method"]
        assert np.isclose(current["precision_at_20_pct"], original["precision_at_20_pct"], atol=1e-8)
        assert round(current["roc_auc"], 3) == round(original["roc_auc"], 3)
    assert np.isclose(metrics["training_cross_validation"]["precision_at_20"],
                      student_receipt["training_cross_validation"]["precision_at_20"], atol=1e-8)
    print("Headline results reproduce the student's script run; both runtime versions are recorded.")
print("Provisional action method: frozen Week 4 rule; no tested model improvement at Precision@20.")



Saved the trained model locally.
Saved aggregate results: capstone_notebook_metrics.json
Headline results reproduce the student's script run; both runtime versions are recorded.
Provisional action method: frozen Week 4 rule; no tested model improvement at Precision@20.


## 5. Limitations

One March decision window, unequal client inventories, retrospective complete-case labels, and unknown historical ingestion times limit generalization. All inputs end March 14 and outcomes start March 18, but the three-day reporting buffer is an assumption without ingestion timestamps. Requiring complete future coverage selects better-tracked pages for evaluation; the past-only scoring population does not require future coverage.

Test clients were absent from model fitting, but their outcomes were previously examined in weekly assignments, so this is a reused evaluation set, not a blind sealed test. June remains untouched. Cross-client performance at one date is not a later-month validation. Only seven test clients contribute Precision@20, and we have not established statistical superiority from the one-slot difference. The two methods' top lists may overlap and errors are not independent.

Scores are not calibrated; rankings identify observed decline risk, not whether editing will help. Correlated derived features make individual coefficients hard to interpret as independent signal strength. No causal effects, algorithm claims, or article-quality conclusions are supported. A refresh recommendation requires human review of measurement, demand, and content context.

## 6. Ranked recommendations

To be completed after the model/rule comparison and skeptical review.

## 7. Artifacts the paper embeds

Executed model comparison and error review above; source: `scripts/train_capstone_model.py`. Aggregate receipts: `work/outputs/capstone_feature_table.json`, `capstone_split.json`, `capstone_model_metrics.json` (student run), and `capstone_notebook_metrics.json` (notebook run). The paper's charts, action playbook, and deployment are next steps. Generated CSV caches and model binaries remain local and ignored.

## Progress and final self-check

- [x] Research question and decision are stated.
- [x] Five features are prepared from past measurements only.
- [x] The target and identifiers are separate from model inputs.
- [x] Data preparation cells executed successfully and their outputs are retained.
- [x] Client-grouped split is complete and matches the frozen baseline.
- [x] Train-only preprocessing and model fitting are complete.
- [x] Model and frozen baseline are compared on the same evaluation rows.
- [ ] Validation, recommendations, and honest limitations are complete.
- [ ] The public research paper includes all nine required sections and FlyRank data credit.
- [ ] The deployed paper URL is saved as one line in `submission/paper_url.txt`.

This is an intermediate capstone checkpoint, not the final submission.